# Telecom Customer Churn Prediction

## Project Overview
This project aims to develop an ensemble machine learning model to predict customer churn using the IBM Telco Customer Churn dataset. It involves data preprocessing, exploratory data analysis, categorical encoding, and class imbalance handling. Random Forest and XGBoost classifiers are trained and evaluated using cross-validation and performance metrics, including ROC-AUC. The project also identifies the top 10 influential features to understand the factors associated with customer churn.

Import Libraries

In [44]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report, auc
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

Load and Inspect the dataset

In [45]:
df = pd.read_csv('../Predictive_Customer_Churn_Classifier/data/Telco-Customer-Churn.csv')

In [46]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [47]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [49]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [50]:
df.isnull().sum().sum()

np.int64(0)

- There are no values within the dataset.

In [51]:
df.duplicated().sum()

np.int64(0)

- There are no duplicates within this dataset.

In [52]:
# Target Distribution
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

- The dataset is not symmetrical as the number of customers that stayed is almost 4 times than those that left.

In [53]:
# Since total charges is considered as a string not numerical, it must be modified 
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors = 'coerce'
)

In [54]:
df['TotalCharges'].isnull().sum()

np.int64(11)

- There are 11 null values within the total charges column.

In [55]:
# Remove the customer ID column as it is unnecessary
df.drop(columns = ['customerID'], inplace = True)

/var/folders/xs/zq2078pd61l68rrzh14wczj00000gp/T/ipykernel_8794/2095266144.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns = ['customerID'], inplace = True)


Exploratory Data Analysis (EDA)

In [ ]:
# Churn Distribution
plt.figure(figsize = (14, 5))
sns.countplot(data=df, x='Churn')

plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')

plt.savefig('visualizations/churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Customer Churn by Contract type
plt.figure(figsize=(9, 5))

sns.countplot(
    data=df,
    x='Contract',
    hue='Churn'
)

plt.title('Customer Churn by Contract Type')
plt.savefig('visualizations/churn_by_contract_type.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Analysis
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(9, 6))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.savefig('visualizations/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

- The total charges and tenure are mostly postively related.

Feature Engineering and Data Splitting

In [59]:
X = df.drop(columns = ['Churn'])

y = df['Churn'].map({
    'No' : 0,
    'Yes' : 1
})

In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Data preprocessing and Encoding

In [61]:
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


/var/folders/xs/zq2078pd61l68rrzh14wczj00000gp/T/ipykernel_8794/77836642.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [62]:
# Numerical preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

In [63]:
# Handle Class Imbalance
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

print(scale_pos_weight)

2.768561872909699


Model Development

In [64]:
# Random Forest Classifer
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [65]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

In [66]:
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [67]:
rf_predictions = rf_pipeline.predict(X_test)

rf_probabilities = rf_pipeline.predict_proba(X_test)[:, 1]

In [68]:
# XGBoost 
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

In [69]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

In [70]:
xgb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [71]:
xgb_predictions = xgb_pipeline.predict(X_test)

xgb_probabilities = xgb_pipeline.predict_proba(X_test)[:, 1]

Cross Validation

In [72]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state= 42
)

In [73]:
# Random Forest with cross validation
rf_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("Random Forest CV scores:", rf_scores)
print("Mean ROC-AUC:", rf_scores.mean())
print("Standard deviation:", rf_scores.std())

Random Forest CV scores: [0.81591012 0.80854459 0.82611927 0.81926874 0.84018878]
Mean ROC-AUC: 0.8220062983253211
Standard deviation: 0.01070882358849864


In [74]:
# XGBoost with cross validation
xgb_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("XGBoost CV scores:", xgb_scores)
print("Mean ROC-AUC:", xgb_scores.mean())
print("Standard deviation:", xgb_scores.std())

XGBoost CV scores: [0.84280734 0.83261435 0.84397468 0.86291867 0.8547759 ]
Mean ROC-AUC: 0.847418187834444
Standard deviation: 0.010456308040533772


Model Evaluation

In [75]:
def evaluate_model(name, y_true, predictions, probabilities):

    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions),
        'Recall': recall_score(y_true, predictions),
        'F1 Score': f1_score(y_true, predictions),
        'ROC-AUC': roc_auc_score(y_true, probabilities)
    }

    print(f"\n{name}")
    print("\nClassification Report:")
    print(classification_report(y_true, predictions))

    return results

In [76]:
# Evaluate Random Forest
results = []
results.append(evaluate_model(
    "Random Forest", y_test, rf_predictions, rf_probabilities
))


Random Forest

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1035
           1       0.55      0.62      0.59       374

    accuracy                           0.77      1409
   macro avg       0.71      0.72      0.71      1409
weighted avg       0.78      0.77      0.77      1409



In [77]:
# Evaluate XGBoost
results.append(evaluate_model(
    "XGBoost", y_test, xgb_predictions, xgb_probabilities
))


XGBoost

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1035
           1       0.53      0.79      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.76      0.77      1409



In [78]:
# Comparison Table
comparison_df = pd.DataFrame(results)

comparison_df


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Random Forest,0.766501,0.553444,0.622995,0.586164,0.821904
1,XGBoost,0.755855,0.526882,0.786096,0.630901,0.844553


In [ ]:
cm = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn']
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')

plt.savefig('visualizations/rf_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, xgb_predictions)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn']
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('XGBoost Confusion Matrix')

plt.savefig('visualizations/xgb_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

Plot ROC-AUC curves

In [ ]:
models = {
    "Random Forest": rf_probabilities,
    "XGBoost": xgb_probabilities,
}

plt.figure(figsize=(9, 7))

for name, probabilities in models.items():

    fpr, tpr, thresholds = roc_curve(
        y_test,
        probabilities
    )

    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {roc_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    label='Random Classifier'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('ROC Curves Comparison')

plt.legend()
plt.grid()
plt.savefig('visualizations/roc_auc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

- An AUC curve of greater 0.8 has been achieved by both Random Forest Classifier and XGboost.

Feature Importance

In [82]:
# RANDOM FOREST
# Get trained Random Forest classifier
rf_classifier = rf_pipeline.named_steps['classifier']

# Retrieve encoded feature names
feature_names = rf_pipeline.named_steps[
    'preprocessor'
].get_feature_names_out()

# Extract importance scores
rf_importances = rf_classifier.feature_importances_

# Create dataframe
rf_feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_importances
})

# Sort and select top 10
rf_top10 = rf_feature_importance.sort_values(
    by='Importance',
    ascending=False
).head(10)

print(rf_top10)

                                Feature  Importance
3                     num__TotalCharges    0.145477
1                           num__tenure    0.129937
2                   num__MonthlyCharges    0.123196
36         cat__Contract_Month-to-month    0.068679
18               cat__OnlineSecurity_No    0.037601
38               cat__Contract_Two year    0.035073
27                  cat__TechSupport_No    0.033026
43  cat__PaymentMethod_Electronic check    0.029245
16     cat__InternetService_Fiber optic    0.028301
21                 cat__OnlineBackup_No    0.018029


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=rf_top10,
    x='Importance',
    y='Feature'
)

plt.title('Top 10 Random Forest Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')

plt.tight_layout()
plt.savefig('visualizations/rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

- For the Random Forest model, Total charges is the most influential feature.


In [84]:
# XGBOOST CLASSIFIER
xgb_classifier = xgb_pipeline.named_steps['classifier']

xgb_importances = xgb_classifier.feature_importances_

xgb_feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_importances
})

xgb_top10 = xgb_feature_importance.sort_values(
    by='Importance',
    ascending=False
).head(10)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=xgb_top10,
    x='Importance',
    y='Feature'
)

plt.title('Top 10 XGBoost Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')

plt.tight_layout()
plt.savefig('visualizations/xgb_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

- For the XGBoost, the contract month to month is the most influential feature.

Saving the best model

In [86]:
joblib.dump(
    xgb_model,
    'models/best_model.pkl'
)

['models/best_model.pkl']

## 8. Project Summary & Key Findings

### 8.1 Model Benchmark Comparison (Test Set)

| Model | Accuracy | Precision | Recall (Churn) | F1-Score | Test ROC-AUC | 5-Fold CV ROC-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **XGBoost Classifier** | 75.59% | 52.69% | **78.61%** | **0.6309** | **0.8446** | **0.8474 ± 0.0105** |
| **Random Forest** | **76.65%** | **55.34%** | 62.30% | 0.5862 | 0.8219 | 0.8220 ± 0.0107 |

---

### 8.2 Key Findings
- **Champion Model Selection:** **XGBoost** was selected and saved as the final production model (`models/best_model.pkl`). In customer churn prediction, **Recall** is the primary metric because failing to identify an at-risk customer (False Negative) is significantly costlier than offering an unnecessary retention perk. XGBoost captured **78.61% of all churners** (vs. 62.30% for Random Forest) while achieving a higher ROC-AUC (**0.8446**).
- **Class Imbalance Strategy:** Handling class imbalance (~26.5% churn rate) using `scale_pos_weight = 2.76` for XGBoost and `class_weight='balanced'` for Random Forest enabled both models to detect minority churn events effectively without synthetic oversampling noise.
- **Top Predictive Drivers:**
  - **Contract Type:** `Month-to-month` contracts are the single strongest risk indicator for churn in XGBoost.
  - **Account Tenure & Charges:** `TotalCharges`, `tenure`, and `MonthlyCharges` are the most influential continuous features in Random Forest. Customers with shorter tenure and higher monthly bills churn at much higher rates.
  - **Service Gaps:** Customers lacking bundled support services (`OnlineSecurity`, `TechSupport`) or enrolled in `Fiber optic` without add-ons showed elevated churn propensity.

---

### 8.3 Actionable Business Recommendations
1. **Incentivize Contract Migrations:** Offer discounted introductory rates or loyalty perks to transition high-risk month-to-month users into 1- or 2-year contracts.
2. **Early-Tenure Onboarding:** Deploy targeted retention interventions during the first 6–12 months of tenure, where customer vulnerability is highest.
3. **Service Bundling:** Provide complimentary or discounted online security and technical support packages to strengthen customer stickiness.